In [13]:
from pyiceberg.catalog import load_catalog
import pandas as pd


In [14]:
catalog = load_catalog(
    "book_library",
    **{
        "type": "sql",
        "uri": "sqlite:///./catalog/iceberg_catalog.db",
        "warehouse": "s3://book-library-lakehouse/bronze",

        "s3.endpoint": "http://127.0.0.1:9000",
        "s3.access-key-id": "nelacopot",
        "s3.secret-access-key": "nelacopot",
        "s3.region": "us-east-1",
        "s3.force-virtual-addressing": "false",

        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO"
    }
)

print("Namespaces:")
print(catalog.list_namespaces())

print("\nMongoDB tabele:")
print(catalog.list_tables("mongodb_atlas"))

print("\nOpen Library tabele:")
print(catalog.list_tables("openlibrary"))

Namespaces:
[('mongodb_atlas',), ('openlibrary',)]

MongoDB tabele:
[('mongodb_atlas', 'authors'), ('mongodb_atlas', 'books')]

Open Library tabele:
[('openlibrary', 'books'), ('openlibrary', 'books__isbn'), ('openlibrary', 'books__language')]


In [15]:
tables_to_check = [
    ("mongodb_atlas", "authors"),
    ("mongodb_atlas", "books"),
    ("openlibrary", "books")
]

for namespace, table_name in tables_to_check:

    print("\n--------------------------------")
    print(f"Tabela: {namespace}.{table_name}")
    print("--------------------------------")

    table = catalog.load_table((namespace, table_name))
    arrow_table = table.scan().to_arrow()

    print("Število zapisov:", arrow_table.num_rows)
    print("Lokacija:", table.location())
    print("Particije:", table.spec())

    snapshot = table.current_snapshot()

    if snapshot:
        print("Snapshot ID:", snapshot.snapshot_id)
    else:
        print("Tabela še nima snapshot-a.")

    display(
        arrow_table.to_pandas(
            types_mapper=pd.ArrowDtype
        ).head()
    )


--------------------------------
Tabela: mongodb_atlas.authors
--------------------------------
Število zapisov: 2006
Lokacija: s3://book-library-lakehouse/bronze/mongodb_atlas/authors
Particije: [
  1000: _ingested_at_day: day(9)
]
Snapshot ID: 5544653622342132227


,_id,first_name,last_name,created_at,updated_at,v,_source_system,_ingestion_type,_ingested_at,_load_id,_dlt_load_id,_dlt_id
0,6ad363ef5a8e469db2debe6a,Ivan,Cankar,2026-10-17 12:02:55.696000+00:00,2026-10-17 12:02:55.696000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,pybSG/Lek79rwg
1,6ad364ef9165d5096ad272f8,Karlee,Romaguera,2026-10-17 12:07:11.120000+00:00,2026-10-17 12:07:11.120000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,WTbPd4+/AWQM+Q
2,6ad364ef9165d5096ad272f9,Dan,Berge-Armstrong,2026-10-17 12:07:11.120000+00:00,2026-10-17 12:07:11.120000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,RNGhIUd+nCZvOA
3,6ad364ef9165d5096ad272fa,Schuyler,Greenfelder,2026-10-17 12:07:11.120000+00:00,2026-10-17 12:07:11.120000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,WBiURzDN6lBZtg
4,6ad364ef9165d5096ad272fb,Van,Conroy,2026-10-17 12:07:11.120000+00:00,2026-10-17 12:07:11.120000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,dhC38RVDXB2wxw



--------------------------------
Tabela: mongodb_atlas.books
--------------------------------
Število zapisov: 9908
Lokacija: s3://book-library-lakehouse/bronze/mongodb_atlas/books
Particije: [
  1000: _ingested_at_day: day(9)
]
Snapshot ID: 6859439240913658252


,_id,title,author,created_at,updated_at,v,_source_system,_ingestion_type,_ingested_at,_load_id,_dlt_load_id,_dlt_id
0,6ad363ef5a8e469db2debe6b,Na klancu,6ad363ef5a8e469db2debe6a,2026-10-17 12:02:55.891000+00:00,2026-10-17 12:02:55.891000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,9ROfRCvehrDdVA
1,6ad364f89165d5096ad27aca,Romeo and Juliet,6ad363ef5a8e469db2debe6a,2026-10-17 12:07:19.852000+00:00,2026-10-17 12:07:19.852000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,TcXBtwbzWb25WQ
2,6ad364f89165d5096ad27acb,East of Eden,6ad364ef9165d5096ad272f8,2026-10-17 12:07:19.852000+00:00,2026-10-17 12:07:19.852000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,e9qpLWwLXhN12w
3,6ad364f89165d5096ad27acc,The Waste Land,6ad364ef9165d5096ad272f9,2026-10-17 12:07:19.852000+00:00,2026-10-17 12:07:19.852000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,yFNiz+EzG4PArA
4,6ad364f89165d5096ad27acd,The Hound of the Baskervilles,6ad364ef9165d5096ad272fa,2026-10-17 12:07:19.852000+00:00,2026-10-17 12:07:19.852000+00:00,0,mongodb_atlas,incremental,2026-11-03 10:26:11.086863+00:00,cdec8787-3910-44c9-9059-b41831128407,1793701575.8403804,duHPx74Uc918+Q



--------------------------------
Tabela: openlibrary.books
--------------------------------
Število zapisov: 5856
Lokacija: s3://book-library-lakehouse/bronze/openlibrary/books
Particije: [
  1000: _ingested_at_day: day(11)
]
Snapshot ID: 7070250783243044529


,edition_count,first_publish_year,key,title,authors,author_ids,openlibrary_id,_source_system,_source_resource,_ingestion_type,_ingested_at,_load_id,_dlt_load_id,_dlt_id
0,23,1999,/works/OL2784125W,Learning Python,Mark Lutz; David Ascher,OL411267A; OL2726848A,/works/OL2784125W,openlibrary,books,full,2026-11-03 10:40:21.714031+00:00,4ef278b4-d161-4cac-8997-847ee19eb7b2,1793702421.9601157,Ng1bn/iA9WtGxg
1,8,2012,/works/OL17422847W,Python For Data Analysis,Wes McKinney,OL7309633A,/works/OL17422847W,openlibrary,books,full,2026-11-03 10:40:21.714031+00:00,4ef278b4-d161-4cac-8997-847ee19eb7b2,1793702421.9601157,5wgUj747dqhq+g
2,6,2015,/works/OL19932156W,Fluent Python,Luciano Ramalho,OL7584543A,/works/OL19932156W,openlibrary,books,full,2026-11-03 10:40:21.714031+00:00,4ef278b4-d161-4cac-8997-847ee19eb7b2,1793702421.9601157,2w90XoCT75h0mw
3,3,2014,/works/OL19547345W,Black Hat Python,Justin Seitz; Tim Arnold,OL6608409A; OL333934A,/works/OL19547345W,openlibrary,books,full,2026-11-03 10:40:21.714031+00:00,4ef278b4-d161-4cac-8997-847ee19eb7b2,1793702421.9601157,yOBfnScTKgQrow
4,7,2002,/works/OL14873289W,Python Cookbook,Alex Martelli; David Ascher; David M. Beazley;...,OL1521655A; OL2726848A; OL35901A; OL4750007A,/works/OL14873289W,openlibrary,books,full,2026-11-03 10:40:21.714031+00:00,4ef278b4-d161-4cac-8997-847ee19eb7b2,1793702421.9601157,2fGDfx9GL8NaQA
